# First Lesson for Data Scientists: From Question → EDA → Baseline

This notebook walks you through:
1. Framing the problem
2. Loading & inspecting data (EDA)
3. Preventing leakage & splitting data
4. Building a simple baseline model
5. Evaluating and writing down learnings

**Dataset**: a tiny synthetic churn dataset (`ds_first_lesson_tiny.csv`).

## 0) Setup
Run this once. If you don't have dependencies installed, run the pip cell below.

In [ ]:
# %pip install pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import pathlib

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (6,4)

## 1) Load the data

In [ ]:
# If the file is alongside this notebook, this will work as-is.
CSV_PATH = pathlib.Path('ds_first_lesson_tiny.csv')
df = pd.read_csv(CSV_PATH)
df.head()

## 2) EDA: size, schema, basic stats, missingness

In [ ]:
df.shape, df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isna().sum()

In [ ]:
# Simple distributions
for col in ['months_active','avg_sessions','avg_spend']:
    df[col].hist(bins=20)
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel('count')
    plt.show()

## 3) Split the data (avoid leakage)
We do **train/test split before** imputation/encoding to prevent training on information from the test set.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['churn'])
train_df.shape, test_df.shape

### 3.1 Impute & encode **using train-only stats**, then transform test
We'll keep this simple with mean imputation and one-hot encoding.

In [ ]:
# Impute numeric columns with train mean
num_cols = ['months_active','avg_sessions','avg_spend']
cat_cols = ['country']

train_imp = train_df.copy()
train_imp[num_cols] = train_imp[num_cols].fillna(train_imp[num_cols].mean())
test_imp = test_df.copy()
test_imp[num_cols] = test_imp[num_cols].fillna(train_imp[num_cols].mean())

# One-hot encode category using train categories
train_enc = pd.get_dummies(train_imp, columns=cat_cols, drop_first=True)
test_enc = pd.get_dummies(test_imp, columns=cat_cols, drop_first=True)

# Align columns (test may miss some columns)
test_enc = test_enc.reindex(columns=train_enc.columns, fill_value=0)

X_train = train_enc.drop(columns=['churn']).values
y_train = train_enc['churn'].values
X_test  = test_enc.drop(columns=['churn']).values
y_test  = test_enc['churn'].values
X_train.shape, X_test.shape

## 4) Train a simple baseline
We'll use Logistic Regression with default settings.

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title('Confusion Matrix (Baseline)')
plt.show()

## 5) Takeaways & next steps
- Document assumptions and data quality issues you found (missingness, skew).
- List potential leakage risks for your real project (time leakage, target encoders).
- Try a couple of simple changes: regularization C, different imputers, or tree models.
- Write down an **error analysis**: which countries or ranges have worse performance?